In [ ]:
### Imports + paths + load data

In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Notebook location: ml/models/research/A4/
SCORES_PATH = Path("../../data/processed/20190108_scores_and_weak_links.csv")
FEATURES_PATH = Path("../../data/processed/AimoScore_WeakLink_big_scores.csv")
REG_PATH = Path("../../data/raw/AimoScore_WeakLink_big_scores.xls")

scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)
df_reg = pd.read_excel(REG_PATH)

print("scores:", scores.shape)
print("big:", big.shape)
print("regression:", df_reg.shape)

display(scores.head(2))
display(big.head(2))
display(df_reg.head(2))


scores: (2096, 17)
big: (2094, 43)
regression: (2094, 43)


,ID,Date,SCORE,ForwardHead,LeftArmFallForward,RightArmFallForward,LeftShoulderElevation,RightShoulderElevation,ExcessiveForwardLean,LeftAsymmetricalWeightShift,RightAsymmetricalWeightShift,LeftKneeMovesInward,RightKneeMovesInward,LeftKneeMovesOutward,RightKneeMovesOutward,LeftHeelRises,RightHeelRises
0,cf0e95ab-d6ec-475a-9cc0-c9444b8dd080.Kinect,2018-07-19,0.612532,0.31,1.34,1.15,0.33,0.49,0.00,0.0,0.08,0.0,0.0,0.0,0.77,0.0,0.0
1,6210cc6d-575d-4e9f-bab1-4fe1da2a30a3.Kinect,2018-07-30,0.388809,1.49,1.42,1.39,0.00,0.00,1.08,0.1,0.00,0.0,0.0,0.0,0.40,0.0,0.0


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


## Common utilities

In [20]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, KFold, RandomizedSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, log_loss,
    mean_squared_error, make_scorer
)
import joblib

from sklearn.linear_model import LinearRegression, Ridge, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor
from catboost import CatBoostClassifier, CatBoostRegressor


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

rmse_scorer = make_scorer(lambda yt, yp: -rmse(yt, yp), greater_is_better=True)  # neg RMSE


def align_proba_to_all_classes(proba, model_classes, all_classes, renormalize=True, eps=1e-12):
    """
    Expand/reorder proba columns so they match all_classes.
    Missing classes get probability 0.
    If renormalize=True, rows are scaled to sum to 1.
    """
    model_classes = list(model_classes)
    all_classes = list(all_classes)

    out = np.zeros((proba.shape[0], len(all_classes)), dtype=float)
    idx = {c: i for i, c in enumerate(all_classes)}

    for j, c in enumerate(model_classes):
        if c in idx:
            out[:, idx[c]] = proba[:, j]

    if renormalize:
        row_sums = out.sum(axis=1, keepdims=True)
        row_sums = np.maximum(row_sums, eps)
        out = out / row_sums

    return out


def summarize_metrics(y_true, y_pred, proba_aligned, all_classes):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "log_loss": log_loss(y_true, proba_aligned, labels=list(all_classes)),
    }


def _final_estimator(model):
    return model.steps[-1][1] if isinstance(model, Pipeline) else model


def _is_xgb_classifier(model):
    return isinstance(_final_estimator(model), XGBClassifier)


def fit_predict_proba_with_optional_encoding(model, Xtr, ytr, Xte, all_classes):
    """
    Returns:
      y_pred_str (np.array of class names)
      proba_aligned (ndarray aligned to all_classes)
    Handles XGBoost label encoding reliably.
    """
    m = clone(model)

    if _is_xgb_classifier(m):
        le = LabelEncoder()
        le.fit(list(all_classes))   # stable mapping across folds
        ytr_enc = le.transform(ytr)

        m.fit(Xtr, ytr_enc)

        proba_raw = m.predict_proba(Xte)
        ypred_enc = m.predict(Xte)
        ypred_str = le.inverse_transform(ypred_enc)

        proba_aligned = align_proba_to_all_classes(proba_raw, le.classes_, all_classes)
        return np.array(ypred_str), proba_aligned

    # Non-XGB: can fit on string labels
    m.fit(Xtr, ytr)
    proba_raw = m.predict_proba(Xte)
    classes = _final_estimator(m).classes_ if hasattr(_final_estimator(m), "classes_") else sorted(pd.unique(ytr))
    proba_aligned = align_proba_to_all_classes(proba_raw, classes, all_classes)
    ypred_str = m.predict(Xte)
    return np.array(ypred_str), proba_aligned


## Regression section

In [21]:
y_reg = df_reg["AimoScore"].astype(float)

X_reg = df_reg.drop(columns=["AimoScore", "EstimatedScore"], errors="ignore")
X_reg = X_reg.apply(pd.to_numeric, errors="coerce")  # keep NaNs for imputers

print("X_reg:", X_reg.shape, "y_reg:", y_reg.shape)
print("Missing values:", int(X_reg.isna().sum().sum()))


X_reg: (2094, 41) y_reg: (2094,)
Missing values: 2094


## Regression: CV compare models

In [22]:
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

reg_models = {
    "LinearRegression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression())
    ]),
    "Ridge": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(random_state=RANDOM_SEED))
    ]),
    "ElasticNet": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(random_state=RANDOM_SEED, max_iter=5000))
    ]),
    "LightGBM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMRegressor(
            random_state=RANDOM_SEED,
            n_estimators=1500,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.9,
            colsample_bytree=0.9,
            n_jobs=-1,
        ))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBRegressor(
            random_state=RANDOM_SEED,
            n_estimators=2000,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            tree_method="hist",
            n_jobs=-1,
        ))
    ])
}

rows = []
for name, model in reg_models.items():
    scores_cv = cross_validate(
        model, X_reg, y_reg,
        cv=cv_reg,
        scoring={
            "neg_rmse": rmse_scorer,
            "neg_mae": "neg_mean_absolute_error",
            "r2": "r2"
        },
        n_jobs=-1,
        return_train_score=False
    )
    rows.append({
        "model": name,
        "rmse_mean": -scores_cv["test_neg_rmse"].mean(),
        "rmse_std": scores_cv["test_neg_rmse"].std(),
        "mae_mean": -scores_cv["test_neg_mae"].mean(),
        "r2_mean": scores_cv["test_r2"].mean(),
    })

reg_summary = pd.DataFrame(rows).sort_values("rmse_mean").reset_index(drop=True)
display(reg_summary)

best_reg_name = reg_summary.iloc[0]["model"]
print("Best regression by CV RMSE:", best_reg_name)


,model,rmse_mean,rmse_std,mae_mean,r2_mean
0,XGBoost,0.122060,0.004619,0.087441,0.732264
1,LightGBM,0.124049,0.005212,0.088355,0.723290
2,Ridge,0.154727,0.005060,0.117800,0.569983
3,LinearRegression,0.154734,0.005067,0.117806,0.569943
4,ElasticNet,0.236465,0.004366,0.193747,-0.001247


Best regression by CV RMSE: XGBoost


### Optional tuning + save

In [31]:
if "ID" in X_reg.columns:
    X_reg = X_reg.drop(columns=["ID"])
    print("Dropped ID from X_reg. New shape:", X_reg.shape)


Dropped ID from X_reg. New shape: (2094, 40)


In [32]:
best_reg_model = reg_models[best_reg_name]

param_dist = None
if best_reg_name == "LightGBM":
    param_dist = {
        "model__num_leaves": [31, 63, 127],
        "model__min_child_samples": [10, 20, 40, 80],
        "model__subsample": [0.7, 0.85, 0.95, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 0.95, 1.0],
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__n_estimators": [800, 1200, 1800],
    }
elif best_reg_name == "XGBoost":
    param_dist = {
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__subsample": [0.7, 0.85, 0.95, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 0.95, 1.0],
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__n_estimators": [800, 1200, 1800],
    }

if param_dist is not None:
    search = RandomizedSearchCV(
        best_reg_model,
        param_distributions=param_dist,
        n_iter=20,
        scoring=rmse_scorer,
        cv=cv_reg,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_reg, y_reg)
    tuned_reg_model = search.best_estimator_
    print("Best neg RMSE:", float(search.best_score_))
else:
    tuned_reg_model = best_reg_model

tuned_reg_model.fit(X_reg, y_reg)

MODEL_DIR = Path("./saved_models")
MODEL_DIR.mkdir(exist_ok=True)

reg_out = MODEL_DIR / "regression_model.joblib"
joblib.dump(tuned_reg_model, reg_out)
print("Saved regression model to:", reg_out.resolve())


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best neg RMSE: -0.12132493801471982
Saved regression model to: C:\Users\kimwo\software engineering\year 4\LP3\4DT907 - Project\machine_learning_project_4dt907\ml\models\research\A4\saved_models\regression_model.joblib


In [33]:
# Cross-validate all regression candidates
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

reg_rows = []
for name, model in reg_models.items():
    scores = cross_validate(
        model, X_reg, y_reg,
        cv=cv_reg,
        scoring={
            "neg_rmse": rmse_scorer,
            "neg_mae": "neg_mean_absolute_error",
            "r2": "r2",
        },
        n_jobs=-1,
        return_train_score=False,
    )
    reg_rows.append({
        "model": name,
        "rmse_mean": -np.mean(scores["test_neg_rmse"]),
        "rmse_std": np.std(scores["test_neg_rmse"]),
        "mae_mean": -np.mean(scores["test_neg_mae"]),
        "r2_mean": np.mean(scores["test_r2"]),
    })

reg_summary = pd.DataFrame(reg_rows).sort_values("rmse_mean").reset_index(drop=True)
display(reg_summary)
best_reg_name = reg_summary.iloc[0]["model"]
print("Best regression model by RMSE:", best_reg_name)


,model,rmse_mean,rmse_std,mae_mean,r2_mean
0,XGBoost,0.122060,0.004619,0.087441,0.732264
1,LightGBM,0.124049,0.005212,0.088355,0.723290
2,Ridge,0.154727,0.005060,0.117800,0.569983
3,LinearRegression,0.154734,0.005067,0.117806,0.569943
4,ElasticNet,0.236465,0.004366,0.193747,-0.001247


Best regression model by RMSE: XGBoost


In [34]:
best_reg_model = reg_models[best_reg_name]

# Small tuning only if boosted model
param_dist = None

if best_reg_name == "LightGBM":
    param_dist = {
        "model__num_leaves": [31, 63, 127],
        "model__min_child_samples": [10, 20, 40, 80],
        "model__subsample": [0.7, 0.85, 0.95, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 0.95, 1.0],
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__n_estimators": [800, 1200, 1800],
    }
elif best_reg_name == "XGBoost":
    param_dist = {
        "model__max_depth": [3, 4, 5, 6, 8],
        "model__subsample": [0.7, 0.85, 0.95, 1.0],
        "model__colsample_bytree": [0.7, 0.85, 0.95, 1.0],
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__n_estimators": [800, 1200, 1800],
    }
elif best_reg_name == "CatBoost":
    param_dist = {
        "model__depth": [6, 8, 10],
        "model__learning_rate": [0.02, 0.05, 0.1],
        "model__iterations": [800, 1200, 1600],
    }

if param_dist is not None:
    search = RandomizedSearchCV(
        best_reg_model,
        param_distributions=param_dist,
        n_iter=20,
        scoring=rmse_scorer,
        cv=cv_reg,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_reg, y_reg)
    tuned_reg_model = search.best_estimator_
    print("Best neg RMSE:", float(search.best_score_))
else:
    tuned_reg_model = best_reg_model

# Fit final on full data
tuned_reg_model.fit(X_reg, y_reg)

# Save locally in A4/
MODEL_DIR = Path("./saved_models")
MODEL_DIR.mkdir(exist_ok=True)
joblib.dump(tuned_reg_model, MODEL_DIR / "regression_model.joblib")
print("Saved regression model to:", (MODEL_DIR / "regression_model.joblib").resolve())


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best neg RMSE: -0.12132493801471982
Saved regression model to: C:\Users\kimwo\software engineering\year 4\LP3\4DT907 - Project\machine_learning_project_4dt907\ml\models\research\A4\saved_models\regression_model.joblib


## Predict weakest link

In [35]:
# --- Classification: Predict WeakestLink ---
scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)

NON_INDICATORS = {"ID", "Date", "SCORE"}
indicator_cols = [c for c in scores.columns if c not in NON_INDICATORS]

DROP_CLASSES = True
DROP_WEAK_LINKS = {
    "LeftHeelRises",
    "RightHeelRises",
    "LeftKneeMovesInward",
    "RightAsymmetricalWeightShift",
}
DROP_TIES = True

if DROP_CLASSES:
    indicator_cols = [c for c in indicator_cols if c not in DROP_WEAK_LINKS]

scores[indicator_cols] = scores[indicator_cols].apply(pd.to_numeric, errors="coerce")

row_max = scores[indicator_cols].max(axis=1)
ties = (scores[indicator_cols].eq(row_max, axis=0).sum(axis=1) > 1)

scores["WeakestLink"] = scores[indicator_cols].idxmax(axis=1)
scores["HasTie"] = ties

scores_labeled = scores.loc[~scores["HasTie"]].copy() if DROP_TIES else scores.copy()
scores_labeled = scores_labeled.dropna(subset=["WeakestLink"])

labels = scores_labeled[["ID", "WeakestLink"]]
data = big.merge(labels, on="ID", how="inner")

angle_cols = [c for c in data.columns if c.startswith("No_") and c.endswith("_Angle_Deviation")]
nasm_cols  = [c for c in data.columns if c.startswith("No_") and c.endswith("_NASM_Deviation")]
feature_cols = sorted(angle_cols) + sorted(nasm_cols)

X_clf = data[feature_cols].apply(pd.to_numeric, errors="coerce")
y_clf = data["WeakestLink"].astype(str)

mask = X_clf.notna().all(axis=1)
X_clf = X_clf.loc[mask].reset_index(drop=True)
y_clf = y_clf.loc[mask].reset_index(drop=True)

all_classes = sorted(y_clf.unique())

print("X_clf:", X_clf.shape, "y_clf:", y_clf.shape)
print("n_classes:", len(all_classes))
print("min class count:", int(y_clf.value_counts().min()))


X_clf: (1724, 38) y_clf: (1724,)
n_classes: 10
min class count: 46


In [36]:
# Build class weights (helps with imbalance)
class_weights = compute_class_weight(class_weight="balanced", classes=np.array(all_classes), y=y_clf)
class_weight_map = {c: w for c, w in zip(all_classes, class_weights)}

def make_catboost_class_weights():
    # CatBoost uses list aligned to class order
    return [class_weight_map[c] for c in all_classes]

# Candidate classifiers
clf_models = {
    "lgbm_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", LGBMClassifier(
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=63,
            subsample=0.9,
            colsample_bytree=0.9,
            class_weight=class_weight_map,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1
        ))
    ]),
    "xgb_balanced": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", XGBClassifier(
            n_estimators=1500,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1
        ))
    ])
}

def fit_predict_proba(model, Xtr, ytr, Xte, all_classes):
    """Fits model, returns (y_pred_labels, proba_aligned). Handles XGBoost label encoding safely."""
    from sklearn.base import clone
    m = clone(model)

    final = m.steps[-1][1] if isinstance(m, Pipeline) else m
    is_xgb = final.__class__.__name__ == "XGBClassifier"

    if is_xgb:
        le = LabelEncoder()
        le.fit(all_classes)  # stable mapping
        ytr_enc = le.transform(ytr)
        m.fit(Xtr, ytr_enc)
        proba = m.predict_proba(Xte)
        ypred_enc = m.predict(Xte)
        ypred = le.inverse_transform(ypred_enc)
        model_classes = le.classes_
        proba_aligned = align_proba_to_all_classes(proba, model_classes, all_classes, renormalize=True)
        return ypred, proba_aligned

    m.fit(Xtr, ytr)
    ypred = m.predict(Xte)
    proba = m.predict_proba(Xte)
    model_classes = list(final.classes_) if hasattr(final, "classes_") else np.unique(ypred)
    proba_aligned = align_proba_to_all_classes(proba, model_classes, all_classes, renormalize=True)
    return ypred, proba_aligned


In [37]:
# Cross-validation for classifiers
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

rows = []
details = {}

for name, model in clf_models.items():
    fold_rows = []
    for fold, (tr, te) in enumerate(skf.split(X_clf, y_clf), start=1):
        Xtr, Xte = X_clf.iloc[tr], X_clf.iloc[te]
        ytr, yte = y_clf.iloc[tr], y_clf.iloc[te]
        ypred, proba = fit_predict_proba(model, Xtr, ytr, Xte, all_classes)
        fold_rows.append({
            "fold": fold,
            "accuracy": accuracy_score(yte, ypred),
            "balanced_accuracy": balanced_accuracy_score(yte, ypred),
            "macro_f1": f1_score(yte, ypred, average="macro", zero_division=0),
            "weighted_f1": f1_score(yte, ypred, average="weighted", zero_division=0),
            "log_loss": log_loss(yte, proba, labels=list(all_classes)),
        })
    df = pd.DataFrame(fold_rows)
    details[name] = df
    rows.append({
        "model": name,
        "weighted_f1_mean": df["weighted_f1"].mean(),
        "weighted_f1_std": df["weighted_f1"].std(),
        "macro_f1_mean": df["macro_f1"].mean(),
        "log_loss_mean": df["log_loss"].mean(),
    })

clf_summary = pd.DataFrame(rows).sort_values("weighted_f1_mean", ascending=False).reset_index(drop=True)
display(clf_summary)
best_clf_name = clf_summary.iloc[0]["model"]
print("Best classifier by CV weighted F1:", best_clf_name)


C:\Users\kimwo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kimwo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kimwo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kimwo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\kimwo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: U

,model,weighted_f1_mean,weighted_f1_std,macro_f1_mean,log_loss_mean
0,xgb_balanced,0.649390,0.024746,0.547339,1.091199
1,lgbm_balanced,0.648592,0.039748,0.547429,1.630336


Best classifier by CV weighted F1: xgb_balanced


In [38]:
from sklearn.preprocessing import LabelEncoder

best_clf = clf_models[best_clf_name]

# We'll save: model + classes + (optional) label_encoder for XGB
MODEL_DIR = Path("./saved_models")
MODEL_DIR.mkdir(exist_ok=True)

clf_out_path = MODEL_DIR / "classification_model.joblib"

# --- If LightGBM is best, you can tune it; if XGB is best, skip tuning and fit with label encoding ---
weighted_f1_scorer = make_scorer(lambda yt, yp: f1_score(yt, yp, average="weighted", zero_division=0))

if best_clf_name.startswith("lgbm"):
    param_dist = {
        "clf__num_leaves": [15, 31, 63, 127],
        "clf__min_child_samples": [10, 20, 40, 80],
        "clf__subsample": [0.7, 0.85, 0.95, 1.0],
        "clf__colsample_bytree": [0.7, 0.85, 0.95, 1.0],
        "clf__learning_rate": [0.01, 0.03, 0.05, 0.08],
        "clf__n_estimators": [300, 500, 800, 1200],
    }

    search = RandomizedSearchCV(
        best_clf,
        param_distributions=param_dist,
        n_iter=25,
        scoring=weighted_f1_scorer,
        cv=skf,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbose=1
    )
    search.fit(X_clf, y_clf)
    tuned_clf = search.best_estimator_
    print("Tuned LGBM best CV weighted F1:", float(search.best_score_))

    tuned_clf.fit(X_clf, y_clf)

    joblib.dump({"model": tuned_clf, "classes": all_classes, "label_encoder": None}, clf_out_path)
    print("Saved classifier to:", clf_out_path.resolve())

elif best_clf_name.startswith("xgb"):
    # XGB must be trained on numeric labels
    le = LabelEncoder()
    le.fit(all_classes)                 # stable order
    y_enc = le.transform(y_clf)

    tuned_clf = best_clf
    tuned_clf.fit(X_clf, y_enc)

    # Save encoder so inference can map predictions back to strings
    joblib.dump({"model": tuned_clf, "classes": all_classes, "label_encoder": le}, clf_out_path)
    print("Saved XGB classifier + label encoder to:", clf_out_path.resolve())

else:
    # fallback (shouldn't happen with your clf_models)
    tuned_clf = best_clf.fit(X_clf, y_clf)
    joblib.dump({"model": tuned_clf, "classes": all_classes, "label_encoder": None}, clf_out_path)
    print("Saved classifier to:", clf_out_path.resolve())


Saved XGB classifier + label encoder to: C:\Users\kimwo\software engineering\year 4\LP3\4DT907 - Project\machine_learning_project_4dt907\ml\models\research\A4\saved_models\classification_model.joblib
